In [63]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery

The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


In [64]:
%%bigquery
create or replace model remote_models.gemini_pro
  remote with connection `projects/auspicious-lead-382321/locations/us/connections/vertex_connection`
  options (endpoint = 'gemini-pro');

Query is running:   0%|          |

""


# Predict the category of a snack

In [65]:
%%bigquery
select * except (data_source, load_time)
from movie_stg.Snack
where name is not null
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein
0,54,Gold Peak Brewed Tea - 20 fl oz,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN
1,79,Banana Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.0,NaN,99.0,NaN
2,64,Pickle,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2790.0,NaN,6.0,NaN,NaN,NaN
3,81,Mango Smoothie 16 fl oz,270.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,72.0,NaN,68.0,NaN
4,86,Strawberry Banana Smoothie 16 fl oz,260.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.0,NaN,62.0,NaN


In [66]:
%%bigquery
declare prompt_query STRING default "Suggest a category for the snack. If the snack has the word smoothie in its name should be put in the Smoothies category. Return the output as json, include only the id and category in the output.\n";
create or replace table movie_stg_ai.category_predictions_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "name", name))) as prompt
      from movie_stg.Snack
      order by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );

Query is running:   0%|          |

""


In [67]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.category_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"{""id"":8,""category"":""Popcorn""}"
1,"```json\n{""id"":97,""category"":""Cookies""}\n```"
2,"```json\n{""id"":22,""category"":""Other""}\n```"
3,"```json\n{""id"":34,""category"":""Other""}\n```"
4,"```json\n{""id"":4,""category"":""Other""}\n```"
...,...
95,"```json\n{""id"":45,""category"":""Sandwiches""}\n```"
96,"{""id"":57,""category"":""Chips""}"
97,"{""id"":20,""category"":""Other""}"
98,"```json\n{""id"":65,""category"":""Other""}\n```"


In [68]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.category_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"{""id"":8,""category"":""Popcorn""}","{""id"":8,""category"":""Popcorn""}"
1,"```json\n{""id"":97,""category"":""Cookies""}\n```","{""id"":97,""category"":""Cookies""}"
2,"```json\n{""id"":22,""category"":""Other""}\n```","{""id"":22,""category"":""Other""}"
3,"```json\n{""id"":34,""category"":""Other""}\n```","{""id"":34,""category"":""Other""}"
4,"```json\n{""id"":4,""category"":""Other""}\n```","{""id"":4,""category"":""Other""}"
...,...,...
95,"```json\n{""id"":45,""category"":""Sandwiches""}\n```","{""id"":45,""category"":""Sandwiches""}"
96,"{""id"":57,""category"":""Chips""}","{""id"":57,""category"":""Chips""}"
97,"{""id"":20,""category"":""Other""}","{""id"":20,""category"":""Other""}"
98,"```json\n{""id"":65,""category"":""Other""}\n```","{""id"":65,""category"":""Other""}"


In [69]:
%%bigquery
create or replace table movie_stg_ai.category_predictions_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.category_predictions_raw

Query is running:   0%|          |

""


In [70]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.category') as category
from movie_stg_ai.category_predictions_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,category
0,64,Other
1,51,Hot Dogs
2,11,Other
3,33,Burgers
4,54,Drinks
...,...,...
95,88,Smoothies
96,95,Milkshakes
97,1,Other
98,75,Salty Snacks


### Adding new category column

In [71]:
%%bigquery
alter table movie_stg.Snack add column category string;

Executing query with job ID: a84ca486-78c2-4b2d-98ff-0d6a69a518a7
Query executing: 0.87s


ERROR:
 400 Column already exists: category at [1:40]; reason: invalidQuery, location: query, message: Column already exists: category at [1:40]

Location: US
Job ID: a84ca486-78c2-4b2d-98ff-0d6a69a518a7



In [72]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id')
  from movie_stg_ai.category_predictions_formatted
    where 1=1


Query is running:   0%|          |

Downloading:   0%|          |

,f0_
0,64
1,51
2,11
3,33
4,54
...,...
95,88
96,95
97,1
98,75


In [73]:
%%bigquery
update movie_stg.Snack set category =
  (select json_value(ml_generate_text_llm_result, '$.category')
   from movie_stg_ai.category_predictions_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


#### Inspect output

In [74]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where category is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,101,Gummy Bears,150.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Candy,false
1,75,Pretzel Bites: Add Salt,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,Salty Snacks,false
2,72,Pretzel Bites: Add Honey Mustard,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,Snacks,true
3,54,Gold Peak Brewed Tea - 20 fl oz,150.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN,Drinks,true
4,61,Buffalo Dipping Sauce,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1650.0,NaN,3.0,NaN,NaN,NaN,Dips,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,80,Mango Peach Real Fruit Smoothie,250.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,5.0,220.0,55.0,2.0,49.0,1.0,Smoothies,true
96,90,Wildberry Real Fruit Smoothie,250.0,0.0,0.0,0.0,NaN,0.0,0.0,NaN,10.0,220.0,55.0,3.0,45.0,1.0,Smoothies,true
97,85,Strawberry Banana Real Fruit Smoothie,250.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,10.0,330.0,59.0,3.0,46.0,1.0,Smoothies,true
98,38,Chicken Tenders 5 Pieces (Fried),150.0,400.0,45.0,7.0,0.0,22.0,13.0,85.0,1600.0,NaN,40.0,3.0,NaN,37.0,Chicken,false


In [75]:
%%bigquery
update movie_stg.Snack
  set data_source = 'amc_ai' where category is not null

Query is running:   0%|          |

""


# Detect whether a snack is healthy or unhealthy

In [76]:
%%bigquery
declare prompt_query STRING default "Detect whether a snack is healthy or unhealthy. A snack is unhealthy if it has high calories or high sugar or high calories_from_fat or hight total_fat or high sodium, or a combination of these. Include the id in the output as well";
create or replace table movie_stg_ai.is_healthy_predictions_raw as
select *
from ML.generate_text(
  model remote_models.gemini_pro,
  (
    select concat(prompt_query, to_json_string(json_object("id", id, "name", name,
                   "category", category, "calories", calories, "calories_from_fat", calories_from_fat, "total_fat", total_fat, "sodium", sodium))) as prompt
    from movie_stg.Snack
    order by id
  ),
  struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
);

Query is running:   0%|          |

""


In [77]:
%%bigquery
select ml_generate_text_llm_result, prompt
from movie_stg_ai.is_healthy_predictions_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,prompt
0,## Snack Analysis: Pepperoni Flatbread Pizza\n...,Detect whether a snack is healthy or unhealthy...
1,"The snack with id 59, Apple Sauce, is consider...",Detect whether a snack is healthy or unhealthy...
2,This snack is considered healthy. \n\nHere's w...,Detect whether a snack is healthy or unhealthy...
3,This snack is unhealthy because it is high in ...,Detect whether a snack is healthy or unhealthy...
4,This snack is considered unhealthy because it ...,Detect whether a snack is healthy or unhealthy...
...,...,...
95,## Snack Health Analysis: Espresso Shake (ID: ...,Detect whether a snack is healthy or unhealthy...
96,"The snack with id 20, Caramel Corn 46 fl oz, i...",Detect whether a snack is healthy or unhealthy...
97,## Snack Health Analysis: Ranch Dripping Sauce...,Detect whether a snack is healthy or unhealthy...
98,## Snack Analysis: Wildberry Real Fruit Smooth...,Detect whether a snack is healthy or unhealthy...


In [78]:
%%bigquery
create or replace table movie_stg_ai.is_healthy_predictions_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.is_healthy_predictions_raw

Query is running:   0%|          |

""


In [79]:
%%bigquery
select * from movie_stg_ai.is_healthy_predictions_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"The snack with id 69, Pretzel Bites (Plain - N..."
1,This snack is considered healthy. It has 250 c...
2,## Snack Analysis: Pepperoni Flatbread PizzaBa...
3,This snack is considered unhealthy because it ...
4,This snack is unhealthy because it is high in ...
...,...
95,This snack is unhealthy because it has high ca...
96,This snack is unhealthy because it is high in ...
97,The Strawberry Smoothie 24 fl oz is considered...
98,This snack is unhealthy because it is high in ...


In [80]:
%%bigquery
alter table movie_stg.Snack add column is_healthy string;

Executing query with job ID: d62de43b-4b1e-426c-9295-9e746703c164
Query executing: 0.82s


ERROR:
 400 Column already exists: is_healthy at [1:40]; reason: invalidQuery, location: query, message: Column already exists: is_healthy at [1:40]

Location: US
Job ID: d62de43b-4b1e-426c-9295-9e746703c164



In [81]:
%%bigquery
update movie_stg.Snack set is_healthy =
  (select json_value(ml_generate_text_llm_result, '$.is_healthy')
   from movie_stg_ai.is_healthy_predictions_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1;

Query is running:   0%|          |

""


In [82]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where is_healthy is not null

Query is running:   0%|          |

Downloading: |          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy


In [83]:
%%bigquery
update movie_stg.Snack
  set data_source = 'amc_ai' where is_healthy is not null

Query is running:   0%|          |

""


# Enrich calories for Snack

#### Test the generate_text function

In [84]:
%%bigquery
declare prompt_query STRING default "Suggest a value for calories for the snack. Snacks that have is_healthy as true should have lower calories than snacks that have is_healthy as false. If the snack already has a value for calories, keep and use that value for that snack. Return the output as json, include only the id and calories in the output.\n";
select *
from ML.generate_text(
  model remote_models.gemini_pro,
  (
    select concat(prompt_query, to_json_string(json_object("id", id, "name", name, "is_healthy", is_healthy, "calories", calories))) as prompt
    from movie_stg.Snack
    order by id
  ),
  struct(TRUE as flatten_json_output)
);

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,ml_generate_text_rai_result,ml_generate_text_status,prompt
0,"```json\n{""calories"": 250.0, ""id"": 0}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
1,"```json\n{""calories"": 150.0, ""id"": 1}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
2,"```json\n{""calories"": 150.0, ""id"": 2}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
3,"```json\n{""calories"": 250.0, ""id"": 3}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
4,"```json\n{""calories"": 150.0, ""id"": 4}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
...,...,...,...,...
95,"```json\n{""calories"": 150.0, ""id"": 97}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
96,"```json\n{""calories"": 150.0, ""id"": 98}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
97,"```json\n{""calories"": 150.0, ""id"": 99}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...
98,"```json\n{""calories"": 150.0, ""id"": 100}\n```","[{""category"":1,""probability"":1,""probability_sc...",,Suggest a value for calories for the snack. Sn...


#### Tweak the prompt and save the output

In [56]:
%%bigquery
declare prompt_query STRING default "Suggest a value for calories for the snack. Snacks that have is_healthy as true should have lower calories than snacks that have is_healthy as false. Also, if the snack has smaller carbohydrates or protein values, then it will have smaller calories. Snacks in the same category should have similar, but not the exact same values for calories. If the snack already has a value for calories, keep and use that value for that snack. Return the output as json, include only the id and calories in the output.\n";
create or replace table movie_stg_ai.calorie_prediction_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "is_healthy", is_healthy, "calories", calories, "carbohydrates", carbohydrates, "protein", protein, "category", category))) as prompt
      from movie_stg.Snack
      order  by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );

Query is running:   0%|          |

""


In [57]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.calorie_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"```json\n{""calories"": 150.0, ""id"": 78}\n```"
1,"```json\n{""calories"": 150.0, ""id"": 39}\n```"
2,"```json\n{""calories"": 150.0, ""carbohydrates"": ..."
3,"```json\n{""id"": 26, ""calories"": 150.0}\n```"
4,"```json\n{""calories"": 150.0, ""id"": 37}\n```"
...,...
95,"```json\n{""calories"": 150.0, ""id"": 65}\n```"
96,"```json\n{""id"": 90, ""calories"": 250.0}\n```"
97,"```json\n{""calories"": 150.0, ""id"": 5}\n```"
98,"```json\n{""calories"": 150.0, ""id"": 13}\n```"


#### Format the output to proper json

In [85]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.calorie_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"```json\n{""calories"": 150.0, ""id"": 78}\n```","{""calories"": 150.0, ""id"": 78}"
1,"```json\n{""calories"": 150.0, ""id"": 39}\n```","{""calories"": 150.0, ""id"": 39}"
2,"```json\n{""calories"": 150.0, ""carbohydrates"": ...","{""calories"": 150.0, ""carbohydrates"": 3.0, ""cat..."
3,"```json\n{""id"": 26, ""calories"": 150.0}\n```","{""id"": 26, ""calories"": 150.0}"
4,"```json\n{""calories"": 150.0, ""id"": 37}\n```","{""calories"": 150.0, ""id"": 37}"
...,...,...
95,"```json\n{""calories"": 150.0, ""id"": 65}\n```","{""calories"": 150.0, ""id"": 65}"
96,"```json\n{""id"": 90, ""calories"": 250.0}\n```","{""id"": 90, ""calories"": 250.0}"
97,"```json\n{""calories"": 150.0, ""id"": 5}\n```","{""calories"": 150.0, ""id"": 5}"
98,"```json\n{""calories"": 150.0, ""id"": 13}\n```","{""calories"": 150.0, ""id"": 13}"


In [58]:
%%bigquery
create or replace table movie_stg_ai.calorie_prediction_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.calorie_prediction_raw

Query is running:   0%|          |

""


#### Update the Snack records with the predicted calories

In [59]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.calories') as calories
from movie_stg_ai.calorie_prediction_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,calories
0,62,150.0
1,66,150.0
2,92,150.0
3,100,150.0
4,64,100.0
...,...,...
95,32,150.0
96,39,150.0
97,88,250.0
98,99,150.0


In [60]:
%%bigquery
update movie_stg.Snack set calories =
  (select max(CAST(JSON_VALUE(ml_generate_text_llm_result, '$.calories') AS FLOAT64))
   from movie_stg_ai.calorie_prediction_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


#### Inspect the output

In [61]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where calories is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,88,Strawberry Smoothie 16 fl oz,250.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.0,NaN,62.0,NaN,Smoothies,true
1,61,Buffalo Dipping Sauce,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1650.0,NaN,3.0,NaN,NaN,NaN,Dips,false
2,78,Banana Smoothie 16 fl oz,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN,66.0,NaN,Smoothies,false
3,86,Strawberry Banana Smoothie 16 fl oz,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.0,NaN,62.0,NaN,Smoothies,false
4,75,Pretzel Bites: Add Salt,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,Salty Snacks,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,17,Plain Popcorn (No Butter) Medium (coconut oil),150.0,230.0,27.0,19.0,NaN,2.5,2.5,NaN,1530.0,350.0,82.0,15.0,NaN,14.0,Popcorn,false
91,41,Curly Fries (Fried),100.0,390.0,44.0,9.0,0.0,13.0,5.0,NaN,1130.0,700.0,54.0,5.0,3.0,5.0,Side Dishes,false
92,25,Cheddar Crunch 85 fl oz,150.0,3010.0,341.0,196.0,NaN,22.0,14.0,105.0,7010.0,1530.0,306.0,5.0,26.0,52.0,Other,false
93,21,Caramel Corn 85 fl oz,150.0,560.0,64.0,43.0,NaN,NaN,NaN,NaN,2980.0,830.0,446.0,NaN,276.0,21.0,Other,false


In [62]:
%%bigquery
update movie_stg.Snack
  set data_source = 'amc_ai' where calories is not null

Query is running:   0%|          |

""


# Enrich total_fat for Snack

#### Tweak the prompt and save the output

In [87]:
%%bigquery
declare prompt_query STRING default "Suggest the total_fat for each snack. Snacks that have is_healthy as true should have lower fat than snacks that have is_healthy as false. Snacks that have higher calories and calories_from_fat tend to have more amount of fat. If the snack already has a value for total_fat, keep and use that value for that snack. Return the output as json, include only the id and total_fat in the output.\n";
create or replace table movie_stg_ai.total_fat_prediction_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "name", name, "calories", calories, "calories_from_fat", calories_from_fat, "is_healthy", is_healthy, "total_fat", total_fat))) as prompt
      from movie_stg.Snack
      order by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );

Query is running:   0%|          |

""


#### Format the output to proper json


In [88]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.total_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"```json\n[\n {\n ""id"": 78,\n ""total_fat..."
1,"```json\n[\n {\n ""id"": 26,\n ""total_fat..."
2,"```json\n[\n {\n ""id"": 37,\n ""total_fat..."
3,"```json\n{""id"": 63, ""total_fat"": 1.0}\n```"
4,"```json\n[\n { ""id"": 66, ""total_fat"": 11.0 },..."
...,...
95,"```json\n[\n {\n ""id"": 72,\n ""total_fat..."
96,"```json\n[\n {\n ""id"": 93,\n ""total_fat..."
97,"```json\n[\n {\n ""id"": 20,\n ""total_fat..."
98,"```json\n[\n {\n ""id"": 65,\n ""total_fat..."


In [89]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.total_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"```json\n[\n {\n ""id"": 78,\n ""total_fat...","[ { ""id"": 78, ""total_fat"": 0.5 }, {..."
1,"```json\n[\n {\n ""id"": 26,\n ""total_fat...","[ { ""id"": 26, ""total_fat"": 15.0 }, { ..."
2,"```json\n[\n {\n ""id"": 37,\n ""total_fat...","[ { ""id"": 37, ""total_fat"": 27.0 }, { ..."
3,"```json\n{""id"": 63, ""total_fat"": 1.0}\n```","{""id"": 63, ""total_fat"": 1.0}"
4,"```json\n[\n { ""id"": 66, ""total_fat"": 11.0 },...","[ { ""id"": 66, ""total_fat"": 11.0 }, { ""id"": 1..."
...,...,...
95,"```json\n[\n {\n ""id"": 72,\n ""total_fat...","[ { ""id"": 72, ""total_fat"": 5.0 }]"
96,"```json\n[\n {\n ""id"": 93,\n ""total_fat...","[ { ""id"": 93, ""total_fat"": 38.0 }, { ..."
97,"```json\n[\n {\n ""id"": 20,\n ""total_fat...","[ { ""id"": 20, ""total_fat"": 35.0 }, { ..."
98,"```json\n[\n {\n ""id"": 65,\n ""total_fat...","[ { ""id"": 65, ""total_fat"": 23.0 }, { ..."


In [90]:
%%bigquery
create or replace table movie_stg_ai.total_fat_prediction_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.total_fat_prediction_raw

Query is running:   0%|          |

""


#### Update the Snack records with the predicted total_fat

In [91]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.total_fat') as total_fat
from movie_stg_ai.total_fat_prediction_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,total_fat
0,None,None
1,74,0.0
2,19,43.0
3,None,None
4,None,None
...,...,...
95,None,None
96,None,None
97,61,10.0
98,None,None


In [92]:
%%bigquery
update movie_stg.Snack set total_fat =
  (select CAST(JSON_VALUE(ml_generate_text_llm_result, '$.total_fat') AS FLOAT64)
   from movie_stg_ai.total_fat_prediction_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


#### Inspect the output

In [93]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where total_fat is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,61,Buffalo Dipping Sauce,150.0,NaN,10.0,NaN,NaN,NaN,NaN,NaN,1650.0,NaN,3.0,NaN,NaN,NaN,Dips,None
1,56,Gold Peak Brewed Tea - 44 fl oz,150.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,40.0,480.0,4.0,NaN,NaN,NaN,Drinks,None
2,74,Pretzel Bites: Add Magic Dust,150.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,Snacks,None
3,99,Funnel Cake Fries,150.0,130.0,15.0,5.0,NaN,NaN,NaN,35.0,380.0,0.0,57.0,NaN,24.0,3.0,Other,None
4,96,Brownie Bites,150.0,190.0,21.0,4.5,NaN,NaN,NaN,45.0,290.0,NaN,66.0,3.0,36.0,6.0,Other,None
5,50,Regular Fries (Baked),150.0,150.0,17.0,4.5,NaN,NaN,NaN,NaN,470.0,920.0,61.0,7.0,NaN,7.0,Sides,None
6,17,Plain Popcorn (No Butter) Medium (coconut oil),150.0,230.0,27.0,19.0,NaN,2.5,2.5,NaN,1530.0,350.0,82.0,15.0,NaN,14.0,Popcorn,None
7,7,Plain Popcorn (No Butter) Medium (canola oil),NaN,240.0,27.0,2.0,0.0,8.0,15.0,NaN,1530.0,350.0,82.0,15.0,NaN,14.0,Popcorn,None
8,92,Cookies & Cream Shake,150.0,480.0,54.0,30.0,1.0,3.0,8.0,135.0,680.0,200.0,131.0,2.0,97.0,15.0,Shakes,None
9,19,Plain Popcorn Tub( No Butter) 170 fl oz (cocon...,100.0,380.0,43.0,31.0,NaN,4.0,4.0,NaN,2400.0,560.0,132.0,25.0,1.0,22.0,Popcorn,None


# **Enrich saturated_fat**

Tweak the prompt and save the output

In [94]:
%%bigquery
declare prompt_query STRING default "Suggest a value for saturated_fat for each snack. Snacks that are made from animal-based foods, dairy products, and tropical oils tend to have high saturated fat. If the snack already has a value for saturated_fat, keep and use that value for that snack. Return the output as json, include only the id and saturated_fat in the output.\n";
create or replace table movie_stg_ai.saturated_fat_prediction_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "name", name, "is_healthy", is_healthy, "calories", calories, "saturated_fat", saturated_fat))) as prompt
      from movie_stg.Snack
      order  by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );


Query is running:   0%|          |

""


In [95]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.saturated_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"```json\n[\n {""id"": 78, ""saturated_fat"": 0.5}..."
1,"```json\n[\n {""id"": 26, ""saturated_fat"": 5.0}..."
2,"```json\n[\n {\n ""id"": 37,\n ""saturated..."
3,"```json\n{""id"": 63, ""saturated_fat"": 0.5}\n```"
4,"```json\n{""id"": 66, ""saturated_fat"": 1.0}\n```"
...,...
95,"```json\n{""id"": 11, ""saturated_fat"": 29.0}\n```"
96,"```json\n[\n {\n ""id"": 1,\n ""saturated_..."
97,"```json\n[\n {\n ""id"": 72,\n ""saturated..."
98,"```json\n{""id"": 93, ""saturated_fat"": 25.0}\n```"


Format the output to proper json

In [96]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.saturated_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"```json\n[\n {""id"": 78, ""saturated_fat"": 0.5}...","[ {""id"": 78, ""saturated_fat"": 0.5}, {""id"": 1..."
1,"```json\n[\n {""id"": 26, ""saturated_fat"": 5.0}...","[ {""id"": 26, ""saturated_fat"": 5.0}, {""id"": 2..."
2,"```json\n[\n {\n ""id"": 37,\n ""saturated...","[ { ""id"": 37, ""saturated_fat"": 4.0 }, ..."
3,"```json\n{""id"": 63, ""saturated_fat"": 0.5}\n```","{""id"": 63, ""saturated_fat"": 0.5}"
4,"```json\n{""id"": 66, ""saturated_fat"": 1.0}\n```","{""id"": 66, ""saturated_fat"": 1.0}"
...,...,...
95,"```json\n{""id"": 11, ""saturated_fat"": 29.0}\n```","{""id"": 11, ""saturated_fat"": 29.0}"
96,"```json\n[\n {\n ""id"": 1,\n ""saturated_...","[ { ""id"": 1, ""saturated_fat"": 2.5 }, ..."
97,"```json\n[\n {\n ""id"": 72,\n ""saturated...","[ { ""id"": 72, ""saturated_fat"": 1.5 }]"
98,"```json\n{""id"": 93, ""saturated_fat"": 25.0}\n```","{""id"": 93, ""saturated_fat"": 25.0}"


In [97]:
%%bigquery
create or replace table movie_stg_ai.saturated_fat_prediction_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.saturated_fat_prediction_raw

Query is running:   0%|          |

""


Update the Snack records with the predicted saturated_fat

In [98]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.saturated_fat') as saturated_fat
from movie_stg_ai.saturated_fat_prediction_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,saturated_fat
0,None,None
1,None,None
2,53,0.0
3,None,None
4,42,18.0
...,...,...
95,75,0.5
96,None,None
97,None,None
98,83,0.5


In [99]:
%%bigquery
update movie_stg.Snack set saturated_fat =
  (select CAST(JSON_VALUE(ml_generate_text_llm_result, '$.saturated_fat') AS FLOAT64)
   from movie_stg_ai.saturated_fat_prediction_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


Inspect the output

In [100]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where saturated_fat is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,30,Boneless Wings (Baked),150.0,160.0,NaN,3.0,NaN,9.0,5.0,65.0,870.0,NaN,21.0,NaN,NaN,27.0,Appetizers,None
1,65,Ranch Dripping Sauce,150.0,200.0,NaN,4.0,NaN,NaN,NaN,20.0,470.0,NaN,4.0,NaN,3.0,1.0,Other,None
2,75,Pretzel Bites: Add Salt,150.0,NaN,NaN,0.5,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,Salty Snacks,None
3,82,Mango Smoothie 24 fl oz,150.0,NaN,NaN,0.5,NaN,NaN,NaN,NaN,75.0,NaN,108.0,NaN,103.0,NaN,Smoothies,None
4,101,Gummy Bears,150.0,50.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Candy,None
5,71,Pretzel Bites: Add Cinnamon Sugar,150.0,NaN,NaN,1.5,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,NaN,Other,None
6,60,BBQ Dipping Sauce,100.0,NaN,NaN,0.5,NaN,NaN,NaN,NaN,740.0,NaN,16.0,NaN,5.0,NaN,Other,None
7,64,Pickle,100.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,2790.0,NaN,6.0,NaN,NaN,NaN,Other,None
8,61,Buffalo Dipping Sauce,150.0,NaN,10.0,1.5,NaN,NaN,NaN,NaN,1650.0,NaN,3.0,NaN,NaN,NaN,Dips,None
9,89,Strawberry Smoothie 24 fl oz,150.0,NaN,NaN,1.5,NaN,NaN,NaN,NaN,NaN,NaN,102.0,NaN,93.0,NaN,Smoothies,None


# **Enrich trans_fat**

In [101]:
%%bigquery
declare prompt_query STRING default "Suggest a value for trans_fat for each snack. Snacks in the category of baked goods, fried foods, and shortenings have high trans fat. Healthy snacks tend to have lower trans_fat than unhealthy snacks. If the snack already has a value for trans_fat, keep and use that value for that snack. Return the output as json, include only the id and trans_fat in the output.\n";
create or replace table movie_stg_ai.trans_fat_prediction_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "name", name, "is_healthy", is_healthy, "calories", calories, "category", category))) as prompt
      from movie_stg.Snack
      order  by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );

Query is running:   0%|          |

""


In [102]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.trans_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"```json\n{""id"": 21, ""trans_fat"": 0.0}\n```"
1,"```json\n{""id"": 58, ""trans_fat"": 0.5}\n```"
2,"```json\n{""id"": 73, ""trans_fat"": 0.1}\n```"
3,"```json\n{""id"": 92, ""trans_fat"": 0.0}\n```"
4,"```json\n{""id"": 40, ""trans_fat"": 0.5}\n```"
...,...
95,"```json\n{""id"": 69, ""trans_fat"": 0.0}\n```"
96,"```json\n{""id"": 11, ""trans_fat"": 0.0}\n```"
97,"```json\n{""id"": 31, ""trans_fat"": 0.5}\n```"
98,"```json\n{""id"": 72, ""trans_fat"": 0.0}\n```"


Format the output to proper json

In [103]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.trans_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"```json\n{""id"": 21, ""trans_fat"": 0.0}\n```","{""id"": 21, ""trans_fat"": 0.0}"
1,"```json\n{""id"": 58, ""trans_fat"": 0.5}\n```","{""id"": 58, ""trans_fat"": 0.5}"
2,"```json\n{""id"": 73, ""trans_fat"": 0.1}\n```","{""id"": 73, ""trans_fat"": 0.1}"
3,"```json\n{""id"": 92, ""trans_fat"": 0.0}\n```","{""id"": 92, ""trans_fat"": 0.0}"
4,"```json\n{""id"": 40, ""trans_fat"": 0.5}\n```","{""id"": 40, ""trans_fat"": 0.5}"
...,...,...
95,"```json\n{""id"": 69, ""trans_fat"": 0.0}\n```","{""id"": 69, ""trans_fat"": 0.0}"
96,"```json\n{""id"": 11, ""trans_fat"": 0.0}\n```","{""id"": 11, ""trans_fat"": 0.0}"
97,"```json\n{""id"": 31, ""trans_fat"": 0.5}\n```","{""id"": 31, ""trans_fat"": 0.5}"
98,"```json\n{""id"": 72, ""trans_fat"": 0.0}\n```","{""id"": 72, ""trans_fat"": 0.0}"


In [104]:
%%bigquery
create or replace table movie_stg_ai.trans_fat_prediction_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.trans_fat_prediction_raw

Query is running:   0%|          |

""


Update the Snack records with the predicted trans_fat

In [105]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.trans_fat') as trans_fat
from movie_stg_ai.trans_fat_prediction_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,trans_fat
0,7,0.0
1,91,0.0
2,39,0.0
3,80,0.0
4,55,0.0
...,...,...
95,40,0.5
96,4,0.0
97,78,0.0
98,52,0.0


In [106]:
%%bigquery
update movie_stg.Snack set trans_fat =
  (select CAST(JSON_VALUE(ml_generate_text_llm_result, '$.trans_fat') AS FLOAT64)
   from movie_stg_ai.trans_fat_prediction_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


Inspect the output

In [107]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where trans_fat is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,101,Gummy Bears,150.0,50.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Candy,None
1,74,Pretzel Bites: Add Magic Dust,150.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,Snacks,None
2,75,Pretzel Bites: Add Salt,150.0,NaN,NaN,0.5,0.0,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,Salty Snacks,None
3,72,Pretzel Bites: Add Honey Mustard,150.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,Snacks,None
4,54,Gold Peak Brewed Tea - 20 fl oz,150.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN,Drinks,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,39,Chili (2.5 Oz),150.0,60.0,NaN,NaN,0.0,NaN,NaN,20.0,620.0,NaN,6.0,1.0,1.0,6.0,Other,None
96,66,Tostitos Cheese Sauce,150.0,100.0,NaN,1.0,0.0,NaN,NaN,5.0,620.0,NaN,15.0,1.0,1.0,1.0,Dips,None
97,26,$1 Dog,150.0,130.0,NaN,NaN,0.0,NaN,NaN,30.0,730.0,NaN,26.0,1.0,4.0,9.0,Other,None
98,42,Spicy Sriracha Dog,150.0,490.0,NaN,18.0,0.0,NaN,1.5,80.0,1630.0,470.0,53.0,1.0,5.0,20.0,Other,None


# **Enrich polyunsat_fat**

In [108]:
%%bigquery
declare prompt_query STRING default "Suggest a value for polyunsat_fat for each snack. Polyunsat_fat is found in plant and animal foods. Snacks that have is_healthy as true should have higher polyunsat_fat than snacks that have is_healthy as false. If the snack already has a value for polyunsat_fat, keep and use that value for that snack. Return the output as json, include only the id and polyunsat_fat in the output.\n";
create or replace table movie_stg_ai.polyunsat_fat_prediction_raw as
  select *
  from ML.generate_text(
    model remote_models.gemini_pro,
    (
      select concat(prompt_query, to_json_string(json_object("id", id, "name", name, "is_healthy", is_healthy, "calories", calories))) as prompt
      from movie_stg.Snack
      order  by id
    ),
    struct(0 as temperature, 8192 as max_output_tokens, 0.0 as top_p, 1 as top_k, TRUE as flatten_json_output)
  );

Query is running:   0%|          |

""


In [109]:
%%bigquery
select ml_generate_text_llm_result
from movie_stg_ai.polyunsat_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result
0,"```json\n{""id"": 48, ""polyunsat_fat"": 2.5}\n```"
1,"```json\n[{""id"": 59, ""polyunsat_fat"": 0.0}]\n```"
2,"```json\n{""id"": 78, ""polyunsat_fat"": 0.0}\n```"
3,"```json\n[\n {\n ""id"": 26,\n ""polyunsat..."
4,"```json\n{""id"": 37, ""polyunsat_fat"": 2.5}\n```..."
...,...
95,"```json\n{""id"": 60, ""polyunsat_fat"": 0.0}\n```"
96,"```json\n{""id"": 35, ""polyunsat_fat"": 0.0}\n```"
97,"```json\n[\n {\n ""id"": 20,\n ""polyunsat..."
98,"```json\n{""id"": 65, ""polyunsat_fat"": 0.0}\n```"


Format the output to proper json

In [110]:
%%bigquery
select ml_generate_text_llm_result, trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as formated_result
from movie_stg_ai.polyunsat_fat_prediction_raw

Query is running:   0%|          |

Downloading:   0%|          |

,ml_generate_text_llm_result,formated_result
0,"```json\n{""id"": 48, ""polyunsat_fat"": 2.5}\n```","{""id"": 48, ""polyunsat_fat"": 2.5}"
1,"```json\n[{""id"": 59, ""polyunsat_fat"": 0.0}]\n```","[{""id"": 59, ""polyunsat_fat"": 0.0}]"
2,"```json\n{""id"": 78, ""polyunsat_fat"": 0.0}\n```","{""id"": 78, ""polyunsat_fat"": 0.0}"
3,"```json\n[\n {\n ""id"": 26,\n ""polyunsat...","[ { ""id"": 26, ""polyunsat_fat"": 0.5 }, ..."
4,"```json\n{""id"": 37, ""polyunsat_fat"": 2.5}\n```...","{""id"": 37, ""polyunsat_fat"": 2.5} ## Justificat..."
...,...,...
95,"```json\n{""id"": 60, ""polyunsat_fat"": 0.0}\n```","{""id"": 60, ""polyunsat_fat"": 0.0}"
96,"```json\n{""id"": 35, ""polyunsat_fat"": 0.0}\n```","{""id"": 35, ""polyunsat_fat"": 0.0}"
97,"```json\n[\n {\n ""id"": 20,\n ""polyunsat...","[ { ""id"": 20, ""polyunsat_fat"": 1.5 }]"
98,"```json\n{""id"": 65, ""polyunsat_fat"": 0.0}\n```","{""id"": 65, ""polyunsat_fat"": 0.0}"


In [111]:
%%bigquery
create or replace table movie_stg_ai.polyunsat_fat_prediction_formatted as
  select trim(replace(replace(replace(ml_generate_text_llm_result, '```json', ''), '```', ''), '\n', '')) as ml_generate_text_llm_result
  from movie_stg_ai.polyunsat_fat_prediction_raw

Query is running:   0%|          |

""


Update the Snack records with the predicted polyunsat_fat

In [112]:
%%bigquery
select json_value(ml_generate_text_llm_result, '$.id') as id,
  json_value(ml_generate_text_llm_result, '$.polyunsat_fat') as polyunsat_fat
from movie_stg_ai.polyunsat_fat_prediction_formatted

Query is running:   0%|          |

Downloading:   0%|          |

,id,polyunsat_fat
0,14,1.5
1,10,0.0
2,90,1.5
3,None,None
4,2,1.5
...,...,...
95,None,None
96,91,1.5
97,54,0.0
98,11,1.5


In [113]:
%%bigquery
update movie_stg.Snack set polyunsat_fat =
  (select CAST(JSON_VALUE(ml_generate_text_llm_result, '$.polyunsat_fat') AS FLOAT64)
   from movie_stg_ai.polyunsat_fat_prediction_formatted
   where CAST(id AS STRING) = json_value(ml_generate_text_llm_result, '$.id'))
where 1=1

Query is running:   0%|          |

""


Inspect the output

In [114]:
%%bigquery
select * except(data_source, load_time)
from movie_stg.Snack
where polyunsat_fat is not null

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,category,is_healthy
0,89,Strawberry Smoothie 24 fl oz,150.0,NaN,NaN,1.5,0.0,0.0,NaN,NaN,NaN,NaN,102.0,NaN,93.0,NaN,Smoothies,None
1,83,Peach Smoothie 16 fl oz,150.0,NaN,NaN,0.5,0.0,0.5,NaN,NaN,20.0,0.0,69.0,NaN,67.0,NaN,Smoothies,None
2,72,Pretzel Bites: Add Honey Mustard,150.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,Snacks,None
3,88,Strawberry Smoothie 16 fl oz,250.0,NaN,NaN,NaN,0.0,1.5,NaN,NaN,NaN,NaN,68.0,NaN,62.0,NaN,Smoothies,None
4,87,Strawberry Banana Smoothie 24 fl oz,150.0,NaN,NaN,0.5,0.0,1.5,NaN,NaN,NaN,NaN,102.0,NaN,93.0,NaN,Smoothies,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,4,House-made Cheese Corn Medium (canola oil),150.0,700.0,NaN,33.0,0.0,10.0,29.0,15.0,680.0,500.0,92.0,15.0,7.0,19.0,Other,None
58,13,House-made Cheese Corn Large (coconut oil),150.0,1000.0,NaN,NaN,0.0,1.5,26.0,25.0,970.0,720.0,131.0,22.0,10.0,27.0,Other,None
59,10,House-made Caramel Corn Large (coconut oil),150.0,540.0,NaN,NaN,0.0,0.0,10.0,70.0,140.0,880.0,508.0,22.0,318.0,20.0,Snacks,None
60,8,Plain Popcorn Refillable Tub(No Butter)190 fl ...,150.0,440.0,NaN,4.0,0.0,10.0,28.0,NaN,2650.0,630.0,148.0,28.0,2.0,25.0,Popcorn,None


# Update data source

In [116]:
%%bigquery
update movie_stg.Snack
  set data_source = 'amc_ai' where 
        category IS NOT NULL 
        OR calories IS NOT NULL
        OR total_fat IS NOT NULL
        OR saturated_fat IS NOT NULL
        OR trans_fat IS NOT NULL
        OR polyunsat_fat IS NOT NULL
        OR carbohydrates IS NOT NULL
        OR is_healthy IS NOT NULL 

Query is running:   0%|          |

""
